In [7]:
import io
import os
import re
import zipfile
import joblib
import nltk
import numpy as np
import pandas as pd
import requests
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# =========================================================
# 1. DESCARGA PROGRAMÁTICA Y EXTRACCIÓN
# =========================================================
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
response = requests.get(url)

if response.status_code == 200:
  print("✓ Descarga exitosa")

  with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("sms_spam_collection")
    print("✓ Extracción exitosa")

  extracted_files = os.listdir("sms_spam_collection")
  print("Archivos extraídos:", extracted_files)

  # =========================================================
  # 2. CARGA DE DATOS EN PANDAS
  # =========================================================
  df = pd.read_csv(
      "sms_spam_collection/SMSSpamCollection",
      sep="\t",
      header=None,
      names=["label", "message"],
  )

  # =========================================================
  # 3. EXPLORACIÓN E INSPECCIÓN INICIAL
  # =========================================================
  print("\n-------------------- HEAD --------------------")
  print(df.head())

  print("\n-------------------- DESCRIBE --------------------")
  print(df.describe())

  print("\n-------------------- INFO --------------------")
  df.info()

  # =========================================================
  # 4. LIMPIEZA DE DATOS: FALTANTES Y DUPLICADOS
  # =========================================================
  print("\n-------------------- COMPROBACIÓN DE NULOS --------------------")
  print("Valores faltantes:\n", df.isnull().sum())

  print("\n-------------------- DUPLICADOS --------------------")
  print("Entradas duplicadas:", df.duplicated().sum())

  # Eliminar duplicados
  df = df.drop_duplicates()
  print(f"Registros tras eliminar duplicados: {len(df)}")

  # =========================================================
  # 5. PREPROCESAMIENTO DE TEXTO (NLTK PASO A PASO)
  # =========================================================
  nltk.download("punkt", quiet=True)
  nltk.download("punkt_tab", quiet=True)
  nltk.download("stopwords", quiet=True)

  print("\n" + "=" * 60)
  print("=== ESTADO INICIAL ANTES DEL PIPELINE NLTK ===")
  print(df.head(5))

  # --- PASO 1: Conversión a Minúsculas ---
  df["message"] = df["message"].str.lower()
  print("\n=== TRAS CONVERSIÓN A MINÚSCULAS ===")
  print(df["message"].head(5))

  # --- PASO 2: Limpieza de Puntuación y Números (Preservando $ y !) ---
  df["message"] = df["message"].apply(
      lambda x: re.sub(r"[^a-z\s$!]", "", str(x))
  )
  print("\n=== TRAS ELIMINACIÓN DE PUNTUACIÓN Y NÚMEROS ===")
  print(df["message"].head(5))

  # --- PASO 3: Tokenización ---
  df["message"] = df["message"].apply(word_tokenize)
  print("\n=== TRAS TOKENIZACIÓN ===")
  print(df["message"].head(5))

  # --- PASO 4: Eliminación de Stop Words ---
  stop_words = set(stopwords.words("english"))
  df["message"] = df["message"].apply(
      lambda tokens: [word for word in tokens if word not in stop_words]
  )
  print("\n=== TRAS ELIMINACIÓN DE STOP WORDS ===")
  print(df["message"].head(5))

  # --- PASO 5: Derivación Regresiva (Stemming) ---
  stemmer = PorterStemmer()
  df["message"] = df["message"].apply(
      lambda tokens: [stemmer.stem(word) for word in tokens]
  )
  print("\n=== TRAS STEMMING ===")
  print(df["message"].head(5))

  # --- PASO 6: Recomposición de Tokens a Cadenas de Texto ---
  df["message"] = df["message"].apply(lambda tokens: " ".join(tokens))
  print("\n=== TRAS RECOMPOSICIÓN A STRING ===")
  print(df["message"].head(5))

  # Variable objetivo binarizada (1 = Spam, 0 = Ham)
  y = df["label"].apply(lambda x: 1 if x == "spam" else 0)

  # =========================================================
  # 6. ENTRenameNTO Y OPTIMIZACIÓN (Pipeline + GridSearchCV)
  # =========================================================
  print("\n" + "=" * 60)
  print("=== 6. ENTRENAMIENTO Y OPTIMIZACIÓN DE HIPERPARÁMETROS ===")

  # Configuración del Vectorizador base
  vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))

  # Encapsulamiento en Pipeline
  pipeline = Pipeline(
      [("vectorizer", vectorizer), ("classifier", MultinomialNB())]
  )

  # Grilla de hiperparámetros para el parámetro de suavizado alpha
  param_grid = {
      "classifier__alpha": [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
  }

  # Búsqueda en grilla con CV de 5 pliegues evaluando el F1-Score
  grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1")
  grid_search.fit(df["message"], y)

  best_model = grid_search.best_estimator_

  # Auditoría del proceso de optimización
  print("✓ Entrenamiento finalizado")
  print(f"Mejores parámetros identificados: {grid_search.best_params_}")
  print(
      "Mejor puntuación F1-Score en Validación Cruzada:"
      f" {grid_search.best_score_:.4f}"
  )

  # =========================================================
  # 7. EVALUACIÓN E INFERENCIA EN NUEVOS DATOS
  # =========================================================
  print("\n" + "=" * 60)
  print("=== 7. INFERENCIA EN NUEVOS MENSAJES ===")


  # Función de preprocesamiento idéntica para mensajes en producción
  def preprocess_message(message):
    message = message.lower()
    message = re.sub(r"[^a-z\s$!]", "", message)
    tokens = word_tokenize(message)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)


  # Mensajes de prueba para evaluación
  new_messages = [
      (
          "Congratulations! You've won a $1000 Walmart gift card. Go to"
          " http://bit.ly/1234 to claim now."
      ),
      "Hey, are we still meeting up for lunch today?",
      (
          "Urgent! Your account has been compromised. Verify your details here:"
          " www.fakebank.com/verify"
      ),
      "Reminder: Your appointment is scheduled for tomorrow at 10am.",
      (
          "FREE entry in a weekly competition to win an iPad. Just text WIN to"
          " 80085 now!"
      ),
  ]

  # Aplicar preprocesamiento a las nuevas entradas
  processed_messages = [preprocess_message(msg) for msg in new_messages]

  # Inferencia de clase y cálculo de probabilidades
  predictions = best_model.predict(processed_messages)
  prediction_probabilities = best_model.predict_proba(processed_messages)

  print("\n-------------------- RESULTADOS DE INFERENCIA --------------------")
  for i, msg in enumerate(new_messages):
    prediction = "Spam" if predictions[i] == 1 else "Not-Spam"
    spam_probability = prediction_probabilities[i][1]
    ham_probability = prediction_probabilities[i][0]

    print(f"Mensaje    : {msg}")
    print(f"Predicción : {prediction}")
    print(
        f"Probabilidades -> Spam: {spam_probability:.4f} | No-Spam:"
        f" {ham_probability:.4f}"
    )
    print("-" * 65)

  # =========================================================
  # 8. PERSISTENCIA Y CARGA DE MODELOS (joblib)
  # =========================================================
  print("\n" + "=" * 60)
  print("=== 8. PERSISTENCIA DE MODELO CON JOBLIB ===")

  model_filename = "spam_detection_model.joblib"
  joblib.dump(best_model, model_filename)
  print(f"✓ Modelo guardado exitosamente en: '{model_filename}'")

  # Verificación de recarga del modelo desde disco
  loaded_model = joblib.load(model_filename)
  reloaded_predictions = loaded_model.predict(processed_messages)

  # Confirmación de coherencia entre el modelo en memoria y el recargado
  assert (predictions == reloaded_predictions).all()
  print(
      "✓ Validación completada: El modelo recargado genera predicciones"
      " idénticas."
  )

else:
  print(
      "Error al descargar el dataset. Código de respuesta:"
      f" {response.status_code}"
  )

✓ Descarga exitosa
✓ Extracción exitosa
Archivos extraídos: ['readme', 'SMSSpamCollection']

-------------------- HEAD --------------------
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

-------------------- DESCRIBE --------------------
       label                 message
count   5572                    5572
unique     2                    5169
top      ham  Sorry, I'll call later
freq    4825                      30

-------------------- INFO --------------------
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572